In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from tqdm import tqdm


# --------------------------------------------------
# Paths
# --------------------------------------------------

PROJECT_ROOT = (
    Path.home()
    / "projects"
    / "chart_data_retrieval"
)

DEPLOT_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "derived"
    / "deplot"
)

MODEL_CACHE_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "model_cache"
)

DEPLOT_E5_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "embeddings"
    / "deplot_e5_base_v2"
)

ORACLE_E5_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "embeddings"
    / "oracle_e5_base_v2"
)

DEPLOT_E5_DIR.mkdir(
    parents=True,
    exist_ok=True
)

deplot_metrics_path = (
    DEPLOT_DATA_DIR
    / "deplot_full_metrics.parquet"
)

query_embeddings_path = (
    ORACLE_E5_DIR
    / "query_embeddings.npy"
)

query_metadata_path = (
    ORACLE_E5_DIR
    / "query_metadata.parquet"
)

required_files = [
    deplot_metrics_path,
    query_embeddings_path,
    query_metadata_path
]

for file_path in required_files:
    print(file_path.name, ":", file_path.exists())

assert all(
    file_path.exists()
    for file_path in required_files
)


# --------------------------------------------------
# Load DePlot tables and existing E5 queries
# --------------------------------------------------

deplot_charts_df = pd.read_parquet(
    deplot_metrics_path
)

query_embeddings = np.load(
    query_embeddings_path
)

query_metadata_df = pd.read_parquet(
    query_metadata_path
)

assert len(deplot_charts_df) == 4104
assert len(query_metadata_df) == 986
assert query_embeddings.shape == (986, 768)

print("\nDePlot charts:", len(deplot_charts_df))
print("Queries:", len(query_metadata_df))
print(
    "Existing query embeddings:",
    query_embeddings.shape
)


# --------------------------------------------------
# Load E5
# --------------------------------------------------

MODEL_NAME = "intfloat/e5-base-v2"

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

e5_model = SentenceTransformer(
    MODEL_NAME,
    device=device,
    cache_folder=str(MODEL_CACHE_DIR)
)

e5_model.max_seq_length = 512
e5_tokenizer = e5_model.tokenizer

print("Model:", MODEL_NAME)
print(
    "Embedding dimension:",
    e5_model.get_sentence_embedding_dimension()
)
print(
    "Maximum sequence length:",
    e5_model.max_seq_length
)


# --------------------------------------------------
# Prepare DePlot passage text
# --------------------------------------------------

deplot_charts_df["e5_passage_text"] = (
    "passage: "
    + deplot_charts_df[
        "deplot_table_text"
    ]
    .fillna("")
    .astype(str)
)


# --------------------------------------------------
# Count tokens
# --------------------------------------------------

def count_e5_tokens(text):
    return (
        len(
            e5_tokenizer.tokenize(text)
        )
        + 2
    )


token_counts = []

for text in tqdm(
    deplot_charts_df["e5_passage_text"],
    desc="Counting DePlot E5 tokens"
):
    token_counts.append(
        count_e5_tokens(text)
    )

deplot_charts_df["e5_token_count"] = (
    token_counts
)

deplot_charts_df["requires_chunking"] = (
    deplot_charts_df["e5_token_count"]
    > e5_model.max_seq_length
)


# --------------------------------------------------
# Token statistics
# --------------------------------------------------

number_requiring_chunking = int(
    deplot_charts_df[
        "requires_chunking"
    ].sum()
)

print("\nDePlot E5 token statistics")

print(
    "Mean:",
    round(
        deplot_charts_df[
            "e5_token_count"
        ].mean(),
        2
    )
)

print(
    "Median:",
    round(
        deplot_charts_df[
            "e5_token_count"
        ].median(),
        2
    )
)

print(
    "95th percentile:",
    round(
        np.percentile(
            deplot_charts_df[
                "e5_token_count"
            ],
            95
        ),
        2
    )
)

print(
    "Maximum:",
    deplot_charts_df[
        "e5_token_count"
    ].max()
)

print(
    "Charts requiring chunking:",
    number_requiring_chunking
)

print(
    "Chunking rate:",
    round(
        number_requiring_chunking
        / len(deplot_charts_df),
        4
    )
)


# --------------------------------------------------
# Breakdown by chart type
# --------------------------------------------------

token_summary_df = (
    deplot_charts_df
    .groupby("chart_type")
    .agg(
        number_of_charts=(
            "chart_id",
            "count"
        ),
        mean_tokens=(
            "e5_token_count",
            "mean"
        ),
        maximum_tokens=(
            "e5_token_count",
            "max"
        ),
        charts_requiring_chunking=(
            "requires_chunking",
            "sum"
        )
    )
)

display(
    token_summary_df.round(2)
)


print("\nLongest DePlot tables")

display(
    deplot_charts_df[
        [
            "chart_id",
            "chart_type",
            "variant_type",
            "e5_token_count",
            "deplot_table_text"
        ]
    ]
    .sort_values(
        "e5_token_count",
        ascending=False
    )
    .head(10)
)

/etc/python/sitecustomize.py:117: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  mod = _original_import(name, globals, locals, fromlist, level)


deplot_full_metrics.parquet : True
query_embeddings.npy : True
query_metadata.parquet : True

DePlot charts: 4104
Queries: 986
Existing query embeddings: (986, 768)

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
Model: intfloat/e5-base-v2
Embedding dimension: 768
Maximum sequence length: 512


Counting DePlot E5 tokens: 100%|██████████| 4104/4104 [00:01<00:00, 3121.48it/s]



DePlot E5 token statistics
Mean: 181.93
Median: 170.0
95th percentile: 358.0
Maximum: 805
Charts requiring chunking: 49
Chunking rate: 0.0119


,number_of_charts,mean_tokens,maximum_tokens,charts_requiring_chunking
chart_type,,,,
h_bar,700,185.14,641,3
line,1400,195.37,805,25
pie,604,108.78,657,2
v_bar,1400,198.42,789,19



Longest DePlot tables


,chart_id,chart_type,variant_type,e5_token_count,deplot_table_text
3364,chartqa_train_two_col_104195__line_green,line,hard_positive,805,title: Share in global GDP adjusted for PPP by...
1017,chartqa_train_two_col_102750__shuffled__vbar_blue,v_bar,hard_negative,789,title: Deal value in billion U.S. dollars by C...
1015,chartqa_train_two_col_102750__vbar_blue,v_bar,anchor,738,title: Deal value in billion U.S. dollars by C...
3017,chartqa_train_two_col_102750__shuffled__line_g...,line,hard_negative,737,title: Deal value in billion U.S. dollars by C...
3830,chartqa_test_two_col_43384__line_green,line,hard_positive,728,title: National debt as percent of GDP by Char...
3831,chartqa_test_two_col_43384__shuffled__line_green,line,hard_negative,724,title: National debt as percent of GDP by Char...
3016,chartqa_train_two_col_102750__line_green,line,hard_positive,688,title: Deal value in billion U.S. dollars by C...
1878,chartqa_test_two_col_103058__shuffled__vbar_blue,v_bar,hard_negative,682,title: Out-of-pocket price in U.S. dollars by ...
1876,chartqa_test_two_col_103058__vbar_blue,v_bar,anchor,682,title: Out-of-pocket price in U.S. dollars by ...
2618,chartqa_train_two_col_21932__shuffled__pie_pal...,pie,hard_negative,657,title: Number of complaints by Characteristic....
